# Exploratory Data Analysis — TMDB 5000 Box Office

Exploring the **processed dataset** used to train the revenue-forecasting DNN.
Dataset: `backend/data/processed/tmdb_5000_processed.csv` (3,165 movies with valid budget & revenue).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
pd.set_option("display.float_format", lambda v: f"{v:,.0f}")

df = pd.read_csv("../backend/data/processed/tmdb_5000_processed.csv")
print(df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing values:\n", df.isna().sum()[df.isna().sum() > 0])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df["budget"], bins=60, color="#8b5cf6")
axes[0].set_title("Budget distribution")

axes[1].hist(np.log1p(df["revenue"]), bins=60, color="#06b6d4")
axes[1].set_title("log1p(revenue) — near Gaussian")

axes[2].scatter(np.log1p(df["budget"]), np.log1p(df["revenue"]), s=8, alpha=0.35, color="#f59e0b")
axes[2].set_title("log(budget) vs log(revenue)")
axes[2].set_xlabel("log1p(budget)"); axes[2].set_ylabel("log1p(revenue)")
plt.tight_layout(); plt.show()

In [ ]:
genres = df["genres"].str.split("|").explode()
g = genres.value_counts().head(15)

genre_rev = df.assign(genres=df["genres"].str.split("|")).explode("genres") \
    .groupby("genres")["revenue"].agg(["count", "mean"]).sort_values("mean", ascending=False)
genre_rev.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
g.sort_values().plot.barh(ax=axes[0], color="#8b5cf6"); axes[0].set_title("Movies per genre")
genre_rev["mean"].head(12).sort_values().plot.barh(ax=axes[1], color="#06b6d4"); axes[1].set_title("Mean revenue by genre")
plt.tight_layout(); plt.show()

In [ ]:
yearly = df.groupby("release_year")["revenue"].agg(["count", "mean", "sum"])
yearly = yearly[yearly["count"] >= 10]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(yearly.index, yearly["mean"] / 1e6, marker="o", ms=4, color="#f59e0b")
ax.set_title("Average revenue by release year ($M)"); ax.set_xlabel("Year"); ax.set_ylabel("Mean revenue ($M)")
plt.show()
print("Peak years:\n", yearly["mean"].sort_values(ascending=False).head(5))

In [ ]:
month_map = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
monthly = df.groupby("release_month")["revenue"].mean().rename(index=month_map)

fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot.bar(ax=ax, color="#8b5cf6")
ax.set_title("Average revenue by release month"); ax.set_ylabel("Mean revenue ($M)")
plt.show()
print("Best release months:\n", monthly.sort_values(ascending=False).head(4))

In [ ]:
num_cols = ["budget", "revenue", "runtime", "popularity", "vote_average", "vote_count", "cast_size",
            "release_year", "release_quarter"]
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, linewidths=0.5)
plt.title("Correlation with revenue")
plt.show()
print("Top correlates with revenue:\n", corr["revenue"].drop("revenue").sort_values(ascending=False))

## Key takeaways

1. **Target is right-skewed** — training on `log1p(revenue)` is essential (the scatter of `log(revenue)` looks approximately linear in `log(budget)`).
2. **Budget is the single strongest numeric feature** (corr ≈ 0.7 with revenue), followed by `popularity` and `vote_count`.
3. **Genre matters**: *Science Fiction*, *Adventure*, *Fantasy* have the highest mean revenue but *Comedy/Drama* dominate in volume.
4. **Release timing**: early December and the summer blockbuster window (May–July) show the highest average revenue.
5. **Old outliers**: several pre-2000 titles earn far above their budget — a good sanity check for the classifier thresholds (FLOP … BLOCKBUSTER).